In [2]:
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split


# ---------------------------------------------------------
# 1. Load & Preprocess CSV Dataset
# ---------------------------------------------------------
def load_pos_csv(csv_path, nrows=None):
  """Reads the Kaggle 'ner_dataset.csv' file containing:

  ['Sentence #', 'Word', 'POS', 'Tag']
  Groups tokens back into complete sentences using the POS column.
  """
  # Load CSV using pandas with latin1 encoding
  df = pd.read_csv(csv_path, encoding='latin1', nrows=nrows)

  # Forward-fill 'Sentence #' so every token gets its parent sentence ID
  df['Sentence #'] = df['Sentence #'].ffill()

  # Drop rows with missing words or POS tags
  df = df.dropna(subset=['Word', 'POS'])

  # Group by 'Sentence #' to extract [(word, pos_tag), ...]
  sentences = []
  for _, group in df.groupby('Sentence #'):
    sentence = [
        (str(row['Word']).lower(), str(row['POS']))
        for _, row in group.iterrows()
    ]
    if sentence:
      sentences.append(sentence)

  return sentences


# Load dataset (Set nrows=None to load the full dataset for max accuracy)
csv_file_path = r'C:\Users\manee\Downloads\NER dataset.csv'
sentences = load_pos_csv(csv_file_path, nrows=50000)
print(f'Successfully loaded {len(sentences)} complete sentences.')

# Split into train and test sets (80% train, 20% test)
train_sents, test_sents = train_test_split(
    sentences, test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 2. Build Vocabularies & State Mappings
# ---------------------------------------------------------
all_tags = sorted(list({tag for sent in train_sents for _, tag in sent}))
all_words = sorted(list({word for sent in train_sents for word, _ in sent}))

# Reserve token for Out-Of-Vocabulary (OOV) test words
UNK_TOKEN = '<UNK>'
all_words.append(UNK_TOKEN)

tag2idx = {tag: i for i, tag in enumerate(all_tags)}
idx2tag = {i: tag for i, tag in enumerate(all_tags)}
word2idx = {word: i for i, word in enumerate(all_words)}

N_states = len(all_tags)
N_vocab = len(all_words)

# ---------------------------------------------------------
# 3 & 4. Count Matrices & Log Probability Conversions
# ---------------------------------------------------------
alpha = 1.0  # Laplace smoothing parameter

initial_counts = np.zeros(N_states)
transition_counts = np.zeros((N_states, N_states))  # [from_state, to_state]
emission_counts = np.zeros((N_states, N_vocab))  # [state, word]

for sent in train_sents:
  first_word, first_tag = sent[0]
  initial_counts[tag2idx[first_tag]] += 1

  for i in range(len(sent)):
    word, tag = sent[i]
    w_idx = word2idx.get(word, word2idx[UNK_TOKEN])
    t_idx = tag2idx[tag]

    emission_counts[t_idx, w_idx] += 1

    if i > 0:
      prev_word, prev_tag = sent[i - 1]
      prev_t_idx = tag2idx[prev_tag]
      transition_counts[prev_t_idx, t_idx] += 1

# Apply Add-1 Smoothing and calculate log-space probabilities
pi_prob = (initial_counts + alpha) / (
    np.sum(initial_counts) + alpha * N_states
)
A_prob = (transition_counts + alpha) / (
    np.sum(transition_counts, axis=1, keepdims=True) + alpha * N_states
)
B_prob = (emission_counts + alpha) / (
    np.sum(emission_counts, axis=1, keepdims=True) + alpha * N_vocab
)

log_pi = np.log(pi_prob)
log_A = np.log(A_prob)
log_B = np.log(B_prob)


# ---------------------------------------------------------
# 5. Vectorized Viterbi Decoding
# ---------------------------------------------------------
def viterbi_decode_vectorized(
    words, log_pi, log_A, log_B, word2idx, tag2idx, idx2tag
):
  T = len(words)
  K = len(tag2idx)

  viterbi = np.zeros((T, K))
  backpointer = np.zeros((T, K), dtype=int)

  # Initialization (t = 0)
  w_0 = word2idx.get(words[0].lower(), word2idx[UNK_TOKEN])
  viterbi[0] = log_pi + log_B[:, w_0]

  # Dynamic programming step (t = 1 to T-1) using matrix broadcasting
  for t in range(1, T):
    w_t = word2idx.get(words[t].lower(), word2idx[UNK_TOKEN])

    # Add previous column of viterbi scores across all state transitions
    trans_probs = viterbi[t - 1][:, np.newaxis] + log_A

    viterbi[t] = np.max(trans_probs, axis=0) + log_B[:, w_t]
    backpointer[t] = np.argmax(trans_probs, axis=0)

  # Backtracking optimal POS tag sequence
  best_path = [0] * T
  best_path[-1] = int(np.argmax(viterbi[T - 1]))

  for t in range(T - 1, 0, -1):
    best_path[t - 1] = backpointer[t, best_path[t]]

  return [idx2tag[i] for i in best_path]


# ---------------------------------------------------------
# 6 & 7. Evaluation on Test Split
# ---------------------------------------------------------
y_true = []
y_pred = []

for sent in test_sents:
  words = [w for w, _ in sent]
  tags = [t for _, t in sent]

  pred_tags = viterbi_decode_vectorized(
      words, log_pi, log_A, log_B, word2idx, tag2idx, idx2tag
  )

  y_true.extend(tags)
  y_pred.extend(pred_tags)

print('\n--- POS Tagger Performance Report ---')
print(f'Accuracy: {accuracy_score(y_true, y_pred):.4f}\n')
print(classification_report(y_true, y_pred, zero_division=0))

# ---------------------------------------------------------
# 8. Unseen Sentences Prediction
# ---------------------------------------------------------
unseen_sentences = [
    'The government announced new economic policies today .'.split(),
    'Global markets reacted positively to the news .'.split(),
    'Engineers are building advanced artificial intelligence software .'.split(),
    'She reads a financial report every morning .'.split(),
    'Thousands of visitors arrived in London last week .'.split(),
]

print('\n--- Predictions on Unseen Sentences ---')
for sentence in unseen_sentences:
  preds = viterbi_decode_vectorized(
      sentence, log_pi, log_A, log_B, word2idx, tag2idx, idx2tag
  )
  tagged_sentence = ' '.join(
      [f'{word}/{tag}' for word, tag in zip(sentence, preds)]
  )
  print(tagged_sentence)

Loaded 43 sentences.


NameError: name 'sample_data' is not defined